# Look up a GNoME structure by formula

Type a formula, run the notebook, get the CIF.

This exists because the GNoME structures are awkward to reach by hand. They live inside a
446 MB zip that is **not indexed by material id** — the archive is keyed by GNoME's own
composition string, so `MoBr5Cl` is stored as `Br5Cl1Mo1.CIF`. Recovering that key means
joining against a 168 MB summary file. This notebook does that once and keeps it in memory.

**What you need** (both gitignored, both local-only):

| file | size | what it is |
|---|---|---|
| `gnome_data/stable_materials_summary.csv` | 168 MB | id ↔ composition ↔ space group, for 554,054 materials |
| `gnome_data/by_composition.zip` | 446 MB | the CIF files themselves |

If they are missing, re-download from the public bucket `gs://gdm_materials_discovery`
(plain HTTPS works, no gcloud needed).

**Run with the project environment**, not the system Python:
`/Users/mac/miniconda3/envs/ml_env/bin/python -m jupyter lab`

## 1 · The one cell you change

In [ ]:
# ============================================================================
#  CHANGE THIS, then Run All.
# ============================================================================
# Any formula works. It is matched on the REDUCED formula, so "Ba2Sr6Sb4H2O",
# "Ba4Sr12Sb8H4O2" and any other multiple of the same ratio all find the same
# material. Case matters for element symbols (Cl not CL).
FORMULA = "Ba2Sr6Sb4H2O"

# Where extracted CIFs are written. One file per match.
OUT_DIR = "cifs/notebook"

# Also report what this project's models predicted for it, if the screen
# outputs are present. Set False to skip (slightly faster).
SHOW_PREDICTIONS = True

## 2 · Setup

Loads the summary once and caches it. The first run takes ~15–30 s because the file is
168 MB; every later lookup in the same session is instant.

In [ ]:
import os, sys, zipfile, warnings
import pandas as pd

# pymatgen warns per-element about missing electronegativity for noble gases
# (He and Ne appear in GNoME). Cosmetic, and it would drown the real output.
warnings.filterwarnings("ignore", category=UserWarning)
from pymatgen.core import Composition, Structure

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SUMMARY_CSV = os.path.join(ROOT, "gnome_data", "stable_materials_summary.csv")
ZIP_PATH    = os.path.join(ROOT, "gnome_data", "by_composition.zip")

for path, hint in [(SUMMARY_CSV, "stable_materials_summary.csv"),
                   (ZIP_PATH,    "by_composition.zip")]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {hint} at {path}\n"
            f"Re-download from the public bucket gs://gdm_materials_discovery "
            f"(anonymous HTTPS works).")

# --- load once, reuse for every lookup in this session ----------------------
# dtype pinned to str deliberately: 34,046 GNoME ids begin with a leading zero
# and 4,632 are all digits, so pandas' default inference turns "000006a8c4"
# into the number 6a8c4-ish nonsense and silently breaks every later join.
if "SUMMARY" not in globals():
    print("loading the GNoME summary (168 MB, first time only) ...")
    SUMMARY = pd.read_csv(SUMMARY_CSV, dtype={"MaterialId": str})
    # A reduced formula for every entry, so lookups are ratio-based rather than
    # depending on how the cell happened to be written down.
    def _reduce(f):
        try:
            return Composition(str(f)).reduced_formula
        except Exception:
            return None
    SUMMARY["reduced"] = SUMMARY["Reduced Formula"].map(_reduce)
    print(f"ready: {len(SUMMARY):,} materials")
else:
    print(f"summary already loaded: {len(SUMMARY):,} materials")

ARCHIVE = zipfile.ZipFile(ZIP_PATH)

## 3 · The lookup

Matches on reduced formula. Several entries can share one — those are genuinely different
structures (polymorphs) with the same composition ratio, and all of them are returned.

In [ ]:
def find(formula, summary=None):
    """Every GNoME entry whose reduced formula matches `formula`."""
    summary = SUMMARY if summary is None else summary
    try:
        target = Composition(formula).reduced_formula
    except Exception as exc:
        raise ValueError(f"'{formula}' is not a formula pymatgen can parse: {exc}")
    hits = summary[summary["reduced"] == target]
    return target, hits


target, hits = find(FORMULA)
print(f"query   : {FORMULA}")
print(f"reduced : {target}")
print(f"matches : {len(hits)}\n")

if len(hits) == 0:
    # A miss is usually a composition GNoME simply does not contain. Offer the
    # nearest thing that IS there - same element set, any ratio - because that
    # is almost always what the person actually wants to see next.
    want = {e.symbol for e in Composition(FORMULA).elements}
    def _same_elements(f):
        try:
            return {e.symbol for e in Composition(str(f)).elements} == want
        except Exception:
            return False
    near = SUMMARY[SUMMARY["Reduced Formula"].map(_same_elements)]
    print(f"Nothing with that exact ratio. {len(near)} entries share the element set "
          f"{{{', '.join(sorted(want))}}}:")
    if len(near):
        print(near[["MaterialId", "Reduced Formula", "NSites",
                    "Space Group", "Crystal System"]].head(15).to_string(index=False))
else:
    cols = ["MaterialId", "Reduced Formula", "NSites", "Volume", "Density",
            "Space Group", "Space Group Number", "Crystal System"]
    print(hits[[c for c in cols if c in hits.columns]].to_string(index=False))

## 4 · Pull the CIF out of the archive

Each match is parsed with pymatgen before it is written, so a file that lands is a file
that opens.

In [ ]:
out_dir = os.path.join(ROOT, OUT_DIR)
os.makedirs(out_dir, exist_ok=True)

structures = {}          # material_id -> pymatgen Structure
written = []

for _, row in hits.iterrows():
    mid, comp = str(row["MaterialId"]), row["Composition"]
    try:
        text = ARCHIVE.read(f"by_composition/{comp}.CIF").decode("utf-8")
        st = Structure.from_str(text, fmt="cif")
    except KeyError:
        print(f"  {mid}: {comp}.CIF is not in the archive")
        continue
    except Exception as exc:
        print(f"  {mid}: {exc.__class__.__name__}: {exc}")
        continue

    safe = "".join(ch if ch.isalnum() else "_" for ch in target)
    fname = f"{safe}__{mid}.cif"
    with open(os.path.join(out_dir, fname), "w", encoding="utf-8") as fh:
        fh.write(text)
    structures[mid] = st
    written.append(fname)

    print(f"\n{mid}  ->  {fname}")
    print(f"  {st.composition.reduced_formula}   {len(st)} sites")
    print(f"  a={st.lattice.a:.4f}  b={st.lattice.b:.4f}  c={st.lattice.c:.4f} A")
    print(f"  alpha={st.lattice.alpha:.2f}  beta={st.lattice.beta:.2f}  gamma={st.lattice.gamma:.2f} deg")
    print(f"  volume={st.volume:.2f} A^3   density={st.density:.4f} g/cm3")
    # Space group from the CIF itself. spglib cannot always determine one, and
    # that is not a reason to fail - the summary's own value is printed above.
    try:
        print(f"  space group (recomputed): {st.get_space_group_info()}")
    except Exception:
        print(f"  space group (recomputed): undetermined by spglib "
              f"(the summary says {row.get('Space Group', '?')})")

print(f"\n{len(written)} file(s) in {OUT_DIR}/")

## 5 · What this project's models predicted for it

Only meaningful for materials that were in the 33,118-candidate screen — GNoME holds
554,054, and the screen kept the subset passing the paper's filters. A blank result here
means the material was screened out, not that anything failed.

**A note on the four numbers:** they are not four independent votes. ALIGNN and the CGCNN
ensemble share matbench training data and derive γ the same way; round 9 is AFLOW-trained
with its own γ head; the tree sees composition only. Round 9 disagrees with the other
three across the whole screen, and that disagreement lives in the moduli.

In [ ]:
if SHOW_PREDICTIONS and len(hits):
    screens = {
        "ALIGNN":    ("results/alignn/57_gnome_screen_alignn.csv", "Kappa_alignn"),
        "CGCNN-ens": ("results/cgcnn/39_gnome_screen_all_gamma.csv", "Kappa_cal_derived_matbench"),
        "round 9":   ("results/cgcnn/39_gnome_screen_all_gamma.csv", "Kappa_r9_gamma"),
        "tree":      ("results/cgcnn/39_gnome_screen_all_gamma.csv", "Kappa_baseline"),
    }
    ids = set(hits["MaterialId"].astype(str))
    cache, found_any = {}, False

    for label, (path, col) in screens.items():
        full = os.path.join(ROOT, path)
        if not os.path.exists(full):
            continue
        if path not in cache:
            cache[path] = pd.read_csv(full, dtype={"material_id": str})
        d = cache[path]
        if col not in d.columns:
            continue
        sub = d[d["material_id"].isin(ids)]
        for _, r in sub.iterrows():
            val = pd.to_numeric(pd.Series([r[col]]), errors="coerce").iloc[0]
            if pd.notna(val):
                found_any = True
                flag = "  <- low-kappa call" if val <= 1.0 else ""
                print(f"  {r['material_id']}  {label:10s} kappa_L = {val:8.3f} W/m/K{flag}")

    if not found_any:
        print("  Not in the screened subset - no predictions for this material.")
        print("  (The screen kept 33,118 of GNoME's 554,054 by the paper's own filters.)")

## 6 · Look at it

A quick 3D scatter of the sites, coloured by element. Deliberately dependency-free — no
`nglview` or `py3Dmol` needed. For real inspection, open the saved `.cif` in VESTA.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

for mid, st in list(structures.items())[:2]:      # at most two, to keep output short
    fig = plt.figure(figsize=(6.4, 5.6))
    ax = fig.add_subplot(111, projection="3d")

    # One colour per element, taken from matplotlib's default cycle so the
    # figure stays readable without hard-coding a palette.
    species = sorted({site.specie.symbol for site in st})
    colours = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    cmap = {s: colours[i % len(colours)] for i, s in enumerate(species)}

    for sym in species:
        pts = [site.coords for site in st if site.specie.symbol == sym]
        xs, ys, zs = zip(*pts)
        # Marker size loosely tracks atomic radius so heavy atoms read as heavy.
        ax.scatter(xs, ys, zs, s=110, color=cmap[sym], edgecolor="white",
                   linewidth=0.6, label=sym, depthshade=True)

    # Draw the unit cell edges so the atoms sit in a box, not in space.
    m = st.lattice.matrix
    origin = [0, 0, 0]
    corners = {(i, j, k): i*m[0] + j*m[1] + k*m[2]
               for i in (0, 1) for j in (0, 1) for k in (0, 1)}
    for (i, j, k), p0 in corners.items():
        for axis, (di, dj, dk) in enumerate([(1, 0, 0), (0, 1, 0), (0, 0, 1)]):
            n = (i + di, j + dj, k + dk)
            if n in corners:
                p1 = corners[n]
                ax.plot(*zip(p0, p1), color="#9aa0a6", lw=0.7)

    ax.set_title(f"{st.composition.reduced_formula}   {mid}\n{len(st)} sites", fontsize=11)
    ax.set_xlabel("x (A)"); ax.set_ylabel("y (A)"); ax.set_zlabel("z (A)")
    ax.legend(loc="upper left", fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

## Notes

**Changing the formula** — edit `FORMULA` in cell 1 and re-run from cell 3. Cell 2's
164 MB load is cached in the session, so subsequent lookups are instant.

**Nothing found?** Most compositions genuinely are not in GNoME. Cell 3 falls back to
listing entries with the same element set in any ratio, which is usually the useful answer.

**Looking one up online** — the Materials Project hosts a
[GNoME Explorer](https://next-gen.materialsproject.org/gnome), but it currently holds a
subset (~45,600 of GNoME's 554,054), and it is not confirmed whether it preserves
DeepMind's original ids. Search it by **formula or chemical system**, not by the
10-character GNoME id. These are ML-predicted structures — most have no ICSD or
experimental record, so finding nothing is expected rather than suspicious.

**Batch extraction** — for a whole candidate list rather than one formula, use
`scripts/cgcnn/62_extract_cifs.py`, which takes any screen CSV plus filters.

**The cell angles may not look like the space group.** `Ba2Sr6Sb4H2O` comes back as
`I4mm` (tetragonal, #107) but with alpha = beta = 97.68 deg. That is not an error: the
archive stores a **primitive** cell, and the primitive cell of a body-centred lattice has
non-90 angles. The symmetry is still I4mm, as the recomputed space group confirms. If you
want the conventional cell — usually what you want to set up a DFT calculation — convert it:

```python
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
conv = SpacegroupAnalyzer(st).get_conventional_standard_structure()
conv.to(filename="conventional.cif")
```

Note the conventional cell has more atoms than the primitive one, so it costs more to compute.